In [1]:
!pip install scikit-survival

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 17.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 85.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 13.3 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored

# --- HÀM XỬ LÝ AN TOÀN KHI NGOẠI SUY THỜI GIAN ---
def safe_prob(fn, t):
    max_time = fn.domain[1]
    if t > max_time:
        return 1 - fn(max_time)
    return 1 - fn(t)

# --- HÀM TÍNH ĐIỂM ĐÁNH GIÁ CỦA CUỘC THI (THÊM MỐC 12H) ---
def evaluate_wids_metric(y_true, risk_scores, surv_funcs):
    c_index = concordance_index_censored(y_true['event'], y_true['time'], risk_scores)[0]
    
    # Bổ sung mốc 12h
    prob_12 = np.array([safe_prob(fn, 12) for fn in surv_funcs])
    prob_24 = np.array([safe_prob(fn, 24) for fn in surv_funcs])
    prob_48 = np.array([safe_prob(fn, 48) for fn in surv_funcs])
    prob_72 = np.array([safe_prob(fn, 72) for fn in surv_funcs])
    
    def brier_at_h(H, prob_h):
        mask_excluded = (~y_true['event']) & (y_true['time'] < H)
        mask_valid = ~mask_excluded
        
        y_valid = y_true[mask_valid]
        p_valid = prob_h[mask_valid]
        
        target = (y_valid['event'] & (y_valid['time'] <= H)).astype(float)
        return np.mean((target - p_valid)**2)

    # Tính Brier cho 4 mốc
    brier_12 = brier_at_h(12, prob_12)
    brier_24 = brier_at_h(24, prob_24)
    brier_48 = brier_at_h(48, prob_48)
    brier_72 = brier_at_h(72, prob_72)
    
    # Giữ nguyên công thức Hybrid Score gốc của BTC (không có 12h)
    wbs = 0.3 * brier_24 + 0.4 * brier_48 + 0.3 * brier_72
    hybrid_score = 0.3 * c_index + 0.7 * (1 - wbs)
    
    return hybrid_score, c_index, wbs, brier_12, brier_24, brier_48, brier_72


# 1. Tải dữ liệu
DATA_DIR = Path("/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26")
train = pd.read_csv(DATA_DIR / "train.csv")
test  = pd.read_csv(DATA_DIR / "test.csv")

# 2. Chuẩn bị Target (y)
y_train = np.empty(len(train), dtype=[('event', bool), ('time', float)])
y_train['event'] = train['event'].astype(bool)
y_train['time'] = train['time_to_hit_hours']

# 3. Chuẩn bị Features (X)
features_to_drop = ['event_id', 'time_to_hit_hours', 'event']
X_train = train.drop(columns=features_to_drop)
X_test = test.drop(columns=['event_id'])

# --- PHỤC HỒI IMPUTER ---
# Nếu không có đoạn này, X_test_imputed bên dưới sẽ báo lỗi undefined
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# 4. Chia tập Validation (Sử dụng X_train_imputed)
X_tr, X_val, y_tr, y_val = train_test_split(X_train_imputed, y_train, test_size=0.2, random_state=42)

# 5. Huấn luyện mô hình
print("Đang huấn luyện mô hình Random Survival Forest...")
rsf = RandomSurvivalForest(n_estimators=50, 
                           min_samples_split=10,
                           min_samples_leaf=15,
                           n_jobs=-1,
                           random_state=42)
rsf.fit(X_tr, y_tr)

# 6. Đánh giá mô hình
print("Đang đánh giá trên tập validation...")
val_risk_scores = rsf.predict(X_val)
val_surv_funcs = rsf.predict_survival_function(X_val)

hybrid, c_idx, wbs, b12, b24, b48, b72 = evaluate_wids_metric(y_val, val_risk_scores, val_surv_funcs)

print("\n--- KẾT QUẢ LOCAL VALIDATION ---")
print(f"Hybrid Score    : {hybrid:.4f}")
print(f"C-index         : {c_idx:.4f}")
print(f"Weighted Brier  : {wbs:.4f}")
print(f"  - Brier@12h   : {b12:.4f}")
print(f"  - Brier@24h   : {b24:.4f}")
print(f"  - Brier@48h   : {b48:.4f}")
print(f"  - Brier@72h   : {b72:.4f}\n")

# 7. Dự đoán và tạo Submission file
print("Đang dự đoán trên tập test...")
test_risk_scores = rsf.predict(X_test_imputed)
test_surv_funcs = rsf.predict_survival_function(X_test_imputed)

# Áp dụng hàm safe_prob cho tập Test với 4 mốc
prob_12_test = np.array([safe_prob(fn, 12) for fn in test_surv_funcs])
prob_24_test = np.array([safe_prob(fn, 24) for fn in test_surv_funcs])
prob_48_test = np.array([safe_prob(fn, 48) for fn in test_surv_funcs])
prob_72_test = np.array([safe_prob(fn, 72) for fn in test_surv_funcs])

submission = pd.DataFrame({
    'event_id': test['event_id'],
    'prob_12h': prob_12_test,
    'prob_24h': prob_24_test,
    'prob_48h': prob_48_test,
    'prob_72h': prob_72_test
})

submission.to_csv("submission.csv", index=False)
print("Đã lưu file submission.csv thành công!")

Đang huấn luyện mô hình Random Survival Forest...
Đang đánh giá trên tập validation...

--- KẾT QUẢ LOCAL VALIDATION ---
Hybrid Score    : 0.8855
C-index         : 0.9279
Weighted Brier  : 0.1326
  - Brier@12h   : 0.1046
  - Brier@24h   : 0.1272
  - Brier@48h   : 0.1326
  - Brier@72h   : 0.1381

Đang dự đoán trên tập test...
Đã lưu file submission.csv thành công!


In [8]:
submission.head()

,event_id,prob_12h,prob_24h,prob_48h,prob_72h
0,10662602,0.129530,0.185279,0.202857,0.352394
1,13353600,0.307482,0.423282,0.447368,0.576195
2,13942327,0.142435,0.205002,0.227550,0.383081
3,16112781,0.322750,0.432409,0.457484,0.590557
4,17132808,0.409362,0.439250,0.442365,0.496957
